# 01 - Decoder training

Trains the task-demand decoder in the two configurations the analysis needs.

**Within-subject cross-validation** (`results_offline_*.pkl`): repeated
stratified 5-fold CV on the calibration block, for three feature sets:
peripheral only, EEG only, and both. Produces the AUCs and the Integrated
Gradients attributions in Figure 1.

**Continuous decoding** (`results_online_simple_physio.pkl`): trains on the
calibration block and slides the decoder across every closed-loop trial,
producing the arousal index used by notebooks 02-04.

Both stages are slow on CPU (several hours for the full cohort). `scripts/`
holds command-line equivalents that are the better route for a full run; set
`SUBJECT_LIMIT` below for a quick check.

In [1]:
import sys
sys.path.insert(0, "..")   # or: pip install -e .. to import `arousal` directly

import pickle
import time

import numpy as np
import torch

from arousal import config as cfg
from arousal import data as D
from arousal import signals as S
from arousal import training as T

DEVICE = T.get_device()
SUBJECT_LIMIT = None      # e.g. 2 to smoke-test on the first two subjects
N_REPETITIONS = 10
OVERWRITE = False

print(f"device: {DEVICE} | torch {torch.__version__}")

device: cuda | torch 2.7.1


## Within-subject cross-validation

Splits are fixed across repetitions (`random_state=13`), so repetitions vary
only the model initialisation and the augmentation draw. Early stopping tracks
training loss; the validation fold is never used for model selection.

Accuracy and AUC are recorded at 16 cumulative epoch lengths, from 32 to 512
samples, which is what lets Figure 1C show attribution as a function of time
before the crossing.

In [2]:
ring = D.load_ring_epochs()
ring = ring[ring["condition"] == cfg.CALIBRATION_CONDITION]
if SUBJECT_LIMIT:
    ring = ring[ring["subj_idx"].isin(cfg.SUBJECTS[:SUBJECT_LIMIT])]
print(f"{len(ring)} calibration epochs, {ring['subj_idx'].nunique()} subjects")

4088 calibration epochs, 16 subjects


In [3]:
def run_config(name):
    """Cross-validate one feature set and cache the result."""
    out = cfg.RESULTS / f"results_offline_simple_{name}.pkl"
    if out.exists() and not OVERWRITE:
        print(f"{out.name} exists; skipping")
        return

    df, modalities = D.prepare_features(ring, name)
    t0 = time.time()
    final_results, imps = T.within_subject_cv(
        df, modalities, DEVICE, n_repetitions=N_REPETITIONS)
    elapsed = time.time() - t0

    per_subject = final_results[:, :, -1, 1].mean(axis=1)
    print(f"{name}: AUC {per_subject.mean()*100:.2f} +/- "
          f"{per_subject.std()/np.sqrt(len(per_subject))*100:.2f} SEM "
          f"({elapsed/60:.1f} min)")
    with open(out, "wb") as f:
        pickle.dump({"final_results": final_results, "imps": imps,
                     "time": elapsed}, f)


for feature_set in D.FEATURE_SETS:
    run_config(feature_set)

results_offline_simple_physio.pkl exists; skipping
results_offline_simple_eeg.pkl exists; skipping
results_offline_simple_all.pkl exists; skipping


## Continuous arousal decoding

For each subject the decoder is trained on the calibration block, then applied
every 16 samples (62.5 ms) to the most recent 512 samples of each closed-loop
trial. The probability is rescaled to 0-100 against the 5th and 95th percentiles
of that subject's *training* distribution, clipped to that range, and smoothed
with an asymmetric IIR filter that lets arousal rise faster than it falls.

The first 256 samples of every trace stay at zero: no full window exists yet.

In [4]:
OUT_ONLINE = cfg.RESULTS / "results_online_simple_physio.pkl"

if OUT_ONLINE.exists() and not OVERWRITE:
    print(f"{OUT_ONLINE.name} exists; skipping")
else:
    online = D.load_ring_epochs_online()
    fixed = D.load_ring_epochs()
    subjects = cfg.SUBJECTS[:SUBJECT_LIMIT] if SUBJECT_LIMIT else cfg.SUBJECTS

    final_results, t0 = [], time.time()
    for s in subjects:
        # Train on the fixed-length calibration epochs ...
        calib = fixed[(fixed["subj_idx"] == s)
                      & (fixed["condition"] == cfg.CALIBRATION_CONDITION)
                      & (fixed["label"] >= 0)]
        X_train = np.stack(calib["data"].to_numpy())[:, cfg.PERIPHERAL_CHANNELS, :]
        y_train = calib["label"].to_numpy()

        # ... and decode the variable-length closed-loop trials.
        subj = online[(online["subj_idx"] == s) & (online["label"] >= 0)]
        trials, _, _ = D.concatenate_trials(
            subj[subj["condition"] != cfg.CALIBRATION_CONDITION],
            channels=cfg.PERIPHERAL_CHANNELS)

        X_train_n, trials_n = S.normalize_train_val(X_train, trials)
        traces = T.decode_subject_trials(
            X_train_n, y_train, trials_n, cfg.MODALITIES_PERIPHERAL, DEVICE,
            n_repetitions=N_REPETITIONS, seed_base=100 * s)
        final_results.append(traces)
        print(f"  S{s:02d}: {len(traces)} trials decoded "
              f"({(time.time()-t0)/60:.1f} min elapsed)")

    with open(OUT_ONLINE, "wb") as f:
        pickle.dump({"final_results": final_results,
                     "time": time.time() - t0}, f)
    print(f"wrote {OUT_ONLINE.name}")

results_online_simple_physio.pkl exists; skipping
